In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Euro NCAP scraper (Data Acquisition project)

Run the cells top to bottom (Runtime > Run all).

1. **Cell 1** is your settings. Put your email in, and leave `LIMIT = 10` for a quick test.
2. **Cell 2** defines the scraper.
3. **Cell 3** runs it. When the 10-car test looks right, set `LIMIT = None` in cell 1 and run again for all ~478 cars (about 15 minutes).
4. **Cell 4** previews the data, **cell 5** downloads the CSV.

How it works: the car URLs come from the public `sitemap.xml`, each page is downloaded politely (robots.txt checked, 1.5 s delay, cached), and the star rating and percentage scores are read from the data embedded in the page. It never calls `/api/`, which robots.txt disallows.

Two rating schemes exist: up to 2025 (adult / child / vulnerable road users / safety assist) and from 2026 (safe driving / crash avoidance / crash protection / post-crash safety). Each car only has the columns for its own scheme. Data source: Euro NCAP (euroncap.com), non-commercial use.

In [ ]:
# Cell 1: settings
YOUR_EMAIL = "C23451962@mytudublin.ie"   # identifies your scraper to the site owner
LIMIT = None                      # test run; set to None for all cars
DELAY_SECONDS = 1.5              # pause between requests (be polite)

In [ ]:
# Cell 2: the scraper
import csv
import re
import time
import urllib.robotparser
from pathlib import Path
from urllib.parse import unquote_plus, urlparse

import requests

BASE = "https://www.euroncap.com"
USER_AGENT = f"Mozilla/5.0 (student data-acquisition project; contact: {YOUR_EMAIL})"
CACHE_DIR = Path("raw_html")
OUT_CSV = Path("euroncap_assessments.csv")

OLD_PILLARS = {
    "adult_occupant_pct": "adultOccupant",
    "child_occupant_pct": "childOccupant",
    "vulnerable_road_users_pct": "vulnerableRoadUsers",
    "safety_assist_pct": "safetyAssist",
}
NEW_PILLARS = {
    "safe_driving_pct": "safeDriving",
    "crash_avoidance_pct": "crashAvoidance",
    "crash_protection_pct": "crashProtection",
    "post_crash_safety_pct": "postCrashSafety",
}
FIELDS = (
    ["url", "make", "model", "protocol_id", "variant_suffix", "rating_year", "protocol_year",
     "scheme", "stars", "tested_model", "body_type", "kerb_weight_kg"]
    + list(OLD_PILLARS) + list(NEW_PILLARS)
)

session = requests.Session()
session.headers["User-Agent"] = USER_AGENT

robots = urllib.robotparser.RobotFileParser()
robots.set_url(f"{BASE}/robots.txt")
robots.parse(session.get(f"{BASE}/robots.txt", timeout=30).text.splitlines())


def polite_get(url):
    if not robots.can_fetch(USER_AGENT, url):
        raise PermissionError(f"robots.txt disallows {url}")
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    time.sleep(DELAY_SECONDS)
    return resp.text


def car_urls():
    """Car safety assessment URLs from the sitemap. IDs starting with a digit are safety
    ratings; IDs starting with 'a' are Assisted Driving gradings (a different dataset)."""
    xml = polite_get(f"{BASE}/sitemap.xml")
    out = []
    for u in re.findall(r"<loc>([^<]+)</loc>", xml):
        parts = urlparse(u).path.strip("/").split("/")
        if len(parts) == 4 and parts[0] == "assessments" and re.match(r"\d{4}", parts[3]):
            out.append(u)
    return out


def get_html(url):
    CACHE_DIR.mkdir(exist_ok=True)
    cache_file = CACHE_DIR / (urlparse(url).path.strip("/").replace("/", "_") + ".html")
    if cache_file.exists():
        return cache_file.read_text(encoding="utf-8")
    html = polite_get(url)
    cache_file.write_text(html, encoding="utf-8")
    return html


def pillar_pct(text, key):
    m = re.search(r'"%s":\{"score":([\d.]+),"maxScore":([\d.]+),"normalisedScore":([\d.]+)' % key, text)
    return float(m.group(3)) if m else None


def star_count(html):
    """Stars are drawn as 5 icons: yellow = filled, grey = empty."""
    i = html.find('data-rating-type="safety"')
    j = html.find('data-component="star-rating"', i) if i >= 0 else -1
    if j < 0:
        return None
    block = html[j:html.find("</div></div>", j)]
    return len(re.findall(r"bg-primary-yellow icon-star", block))


def first(pattern, text):
    m = re.search(pattern, text)
    return m.group(1) if m else None


def parse(url, html):
    text = html.replace('\\"', '"')  # page data is embedded as escaped JSON
    _, make, model, pid = urlparse(url).path.strip("/").split("/")
    row = {
        "url": url,
        "make": unquote_plus(make),
        "model": unquote_plus(model),
        "protocol_id": re.match(r"\d+", pid).group(0),
        "variant_suffix": re.sub(r"^\d+", "", pid),
        "rating_year": first(r'"ratingYear":"?(\d{4})', text),
        "protocol_year": first(r'"protocolYear":"?(\d{4})', text),
        "stars": star_count(html),
        "tested_model": first(r'"testedModel":"([^"]*)"', text),
        "body_type": first(r'"bodyType":\{"code":"([^"]+)"', text),
        "kerb_weight_kg": first(r'"kerbWeight":"?(\d+)', text),
    }
    for col, key in {**OLD_PILLARS, **NEW_PILLARS}.items():
        row[col] = pillar_pct(text, key)
    row["scheme"] = "2026" if row["safe_driving_pct"] is not None else "pre-2026"
    return row


def main(limit=None):
    urls = car_urls()
    if limit:
        urls = urls[:limit]
    print(f"{len(urls)} assessment pages to process")
    rows, failed = [], []
    for n, url in enumerate(urls, 1):
        try:
            rows.append(parse(url, get_html(url)))
        except Exception as exc:
            failed.append((url, str(exc)))
        if n % 25 == 0:
            print(f"  {n}/{len(urls)}")
    with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows(rows)
    print(f"Wrote {len(rows)} rows to {OUT_CSV}")
    for u, err in failed:
        print("FAILED:", u, err)

In [ ]:
# Cell 3: run it
main(LIMIT)

478 assessment pages to process
  25/478
  50/478
  75/478
  100/478
  125/478
  150/478
  175/478
  200/478
  225/478
  250/478
  275/478
  300/478
  325/478
  350/478
  375/478
  400/478


In [ ]:
# Cell 4: preview
import pandas as pd
df = pd.read_csv(OUT_CSV)
print(df.shape)
print(df["scheme"].value_counts())
df.head(10)

(10, 20)
scheme
pre-2026    10
Name: count, dtype: int64


,url,make,model,protocol_id,variant_suffix,rating_year,protocol_year,scheme,stars,tested_model,body_type,kerb_weight_kg,adult_occupant_pct,child_occupant_pct,vulnerable_road_users_pct,safety_assist_pct,safe_driving_pct,crash_avoidance_pct,crash_protection_pct,post_crash_safety_pct
0,https://www.euroncap.com/assessments/ford/must...,ford,mustang,652,NaN,2017,2017,pre-2026,2,"Ford Mustang 5.0 Fastback, LHD",COUPE,1768,72.0,32.0,64.0,16.0,NaN,NaN,NaN,NaN
1,https://www.euroncap.com/assessments/ford/must...,ford,mustang,652,ra,2017,2017,pre-2026,3,"Ford Mustang 5.0 Fastback, LHD",COUPE,1768,72.0,32.0,78.0,61.0,NaN,NaN,NaN,NaN
2,https://www.euroncap.com/assessments/ford/ka+/...,ford,ka,658,NaN,2017,2017,pre-2026,3,"Ford Ka+ 1.2, LHD",HATCHBACK,1059,73.0,61.0,57.0,29.0,NaN,NaN,NaN,NaN
3,https://www.euroncap.com/assessments/volvo/s90...,volvo,s90,659,NaN,2017,2017,pre-2026,5,"Volvo S90 D4 'Momentum', LHD",SEDAN,1828,95.0,80.0,76.0,93.0,NaN,NaN,NaN,NaN
4,https://www.euroncap.com/assessments/volvo/v90...,volvo,v90,659,pm,2017,2017,pre-2026,5,"Volvo S90 D4 'Momentum', LHD",NaN,1828,95.0,80.0,76.0,93.0,NaN,NaN,NaN,NaN
5,https://www.euroncap.com/assessments/audi/q5/0...,audi,q5,660,NaN,2017,2017,pre-2026,5,"Audi Q5 2.0 TDI S tronic (140 kW), LHD",NaN,1805,93.0,86.0,73.0,58.0,NaN,NaN,NaN,NaN
6,https://www.euroncap.com/assessments/citroen/c...,citroen,c3,661,NaN,2017,2017,pre-2026,4,"Citroën C3 1,2 PureTech 82 FEEL, LHD",NaN,1072,88.0,83.0,59.0,58.0,NaN,NaN,NaN,NaN
7,https://www.euroncap.com/assessments/fiat/500/...,fiat,500,662,NaN,2017,2017,pre-2026,3,"Fiat 500 1.2 Pop, LHD",HATCHBACK,865,66.0,49.0,53.0,27.0,NaN,NaN,NaN,NaN
8,https://www.euroncap.com/assessments/toyota/c-...,toyota,c-hr,663,NaN,2017,2017,pre-2026,5,Toyota C-HR Hybrid,SUV,1470,95.0,77.0,76.0,78.0,NaN,NaN,NaN,NaN
9,https://www.euroncap.com/assessments/land+rove...,land rover,discovery,664,NaN,2017,2017,pre-2026,5,"Land Rover Discovery, 2.0 diesel HSE, RHD",SUV,2390,90.0,80.0,75.0,73.0,NaN,NaN,NaN,NaN


In [ ]:
# Cell 5: download the CSV to your computer
from google.colab import files
files.download(str(OUT_CSV))